## 1 -  Importing Libraries

In [9]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

## 2 - Reading Data

In [10]:
data = pd.read_parquet('C:/1 - MyData/1 - Study Santha/z - Projects/BuyProof (AI Product Analyzer)/BuyProof/data/small_data.parquet')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             300000 non-null  float64
 1   review_title       300000 non-null  object 
 2   review_text        300000 non-null  object 
 3   parent_asin        300000 non-null  object 
 4   timestamp          300000 non-null  int64  
 5   helpful_vote       300000 non-null  int64  
 6   verified_purchase  300000 non-null  bool   
 7   category           300000 non-null  object 
 8   product_title      225974 non-null  object 
 9   average_rating     229457 non-null  float64
 10  rating_number      229171 non-null  float64
 11  features           225974 non-null  object 
 12  store              223283 non-null  object 
dtypes: bool(1), float64(3), int64(2), object(7)
memory usage: 27.8+ MB


# 3 - Basic EDA

### 3.1 Info about the data

In [17]:
data.shape
data.dtypes
data.info()
print(data.isnull().sum())
print(data.isnull().mean() * 100)   # % missing per column
data.duplicated(subset=data.columns.difference(['features'])).sum()

<class 'pandas.core.frame.DataFrame'>
Index: 299716 entries, 0 to 299999
Data columns (total 13 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             299716 non-null  float64
 1   review_title       299716 non-null  object 
 2   review_text        299716 non-null  object 
 3   parent_asin        299716 non-null  object 
 4   timestamp          299716 non-null  int64  
 5   helpful_vote       299716 non-null  int64  
 6   verified_purchase  299716 non-null  bool   
 7   category           299716 non-null  object 
 8   product_title      225709 non-null  object 
 9   average_rating     229192 non-null  float64
 10  rating_number      228906 non-null  float64
 11  features           225709 non-null  object 
 12  store              223021 non-null  object 
dtypes: bool(1), float64(3), int64(2), object(7)
memory usage: 30.0+ MB
rating                   0
review_title             0
review_text              0
parent

np.int64(0)

In [18]:
data['rating'].value_counts().sort_index()
data['rating'].describe()

count    299716.000000
mean          4.315329
std           1.167936
min           1.000000
25%           4.000000
50%           5.000000
75%           5.000000
max           5.000000
Name: rating, dtype: float64

In [20]:
print(data['review_text'].str.len().describe())
data['review_text'].isnull().sum()

count    299716.000000
mean        343.814961
std         576.627784
min           0.000000
25%          59.000000
50%         159.000000
75%         394.000000
max       23991.000000
Name: review_text, dtype: float64


np.int64(0)

In [23]:
print(data['category'].nunique())
print(data['category'].value_counts().head(20))

30
category
Toys_and_Games                 10000
Clothing_Shoes_and_Jewelry     10000
Arts_Crafts_and_Sewing         10000
Movies_and_TV                  10000
Electronics                    10000
Sports_and_Outdoors            10000
Office_Products                10000
Home_and_Kitchen               10000
Books                          10000
Appliances                      9999
Amazon_Fashion                  9998
Software                        9998
Baby_Products                   9998
Cell_Phones_and_Accessories     9998
CDs_and_Vinyl                   9996
Health_and_Household            9996
Industrial_and_Scientific       9996
Grocery_and_Gourmet_Food        9995
Beauty_and_Personal_Care        9994
Automotive                      9994
Name: count, dtype: int64


In [24]:
print(data['verified_purchase'].value_counts(normalize=True))

verified_purchase
True     0.727605
False    0.272395
Name: proportion, dtype: float64


In [25]:
print( data['features'].apply(type).value_counts())
print( data['timestamp'].dtype)

features
<class 'numpy.ndarray'>    225709
<class 'NoneType'>          74007
Name: count, dtype: int64
int64


In [26]:
print(  data['verified_purchase'].value_counts(normalize=True))

verified_purchase
True     0.727605
False    0.272395
Name: proportion, dtype: float64


In [29]:
print(data[['helpful_vote', 'rating_number']].describe())
print( data['helpful_vote'].value_counts().head(10))

        helpful_vote  rating_number
count  299716.000000   2.289060e+05
mean        1.528964   1.128658e+04
std        11.259532   5.679601e+04
min         0.000000   1.000000e+00
25%         0.000000   8.300000e+01
50%         0.000000   5.900000e+02
75%         1.000000   3.789000e+03
max      1450.000000   1.898759e+06
helpful_vote
0    210115
1     41866
2     15443
3      8366
4      5071
5      3413
6      2447
7      1826
8      1453
9      1106
Name: count, dtype: int64


In [30]:
data[['rating', 'average_rating', 'rating_number', 'helpful_vote']].corr()

,rating,average_rating,rating_number,helpful_vote
rating,1.000000,0.307381,0.017376,-0.036652
average_rating,0.307381,1.000000,0.088966,-0.033683
rating_number,0.017376,0.088966,1.000000,-0.006785
helpful_vote,-0.036652,-0.033683,-0.006785,1.000000


In [32]:
print(data.groupby('verified_purchase')['rating'].mean())
print(data.groupby('category')['rating'].mean().sort_values())

verified_purchase
False    4.323404
True     4.312305
Name: rating, dtype: float64
category
Subscription_Boxes             3.783795
Software                       3.848170
Amazon_Fashion                 4.107421
All_Beauty                     4.117252
Health_and_Personal_Care       4.145064
Beauty_and_Personal_Care       4.166200
Cell_Phones_and_Accessories    4.191138
Pet_Supplies                   4.231040
Health_and_Household           4.241297
Patio_Lawn_and_Garden          4.255905
Movies_and_TV                  4.264600
Grocery_and_Gourmet_Food       4.276638
Automotive                     4.312387
Electronics                    4.316800
Clothing_Shoes_and_Jewelry     4.358200
Books                          4.359100
Appliances                     4.377638
Baby_Products                  4.378776
Kindle_Store                   4.385724
Tools_and_Home_Improvement     4.395295
Sports_and_Outdoors            4.402900
Industrial_and_Scientific      4.409864
Home_and_Kitchen            